# RouteTune Phase 3: verified QLoRA execution
Run cells in order on a Colab NVIDIA GPU (T4 or better). Upload a ZIP of the current project. The notebook does not assume the GitHub repository is public. Training and held-out evaluation fail closed when evidence is missing.

In [ ]:
import platform, subprocess, sys
print(sys.version)
print(platform.platform())
subprocess.run(["nvidia-smi"], check=True)

Upload one project ZIP made from the current `D:\\Fun\\RouteTune` worktree. This extracts only paths contained by the destination and selects the single directory containing `pyproject.toml`.

In [ ]:
from google.colab import files
from pathlib import Path
import os, zipfile

uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError("Upload exactly one RouteTune project ZIP")
archive = Path("/content") / next(iter(uploaded))
source_root = Path("/content/routetune-source")
source_root.mkdir(exist_ok=False)
with zipfile.ZipFile(archive) as bundle:
    for member in bundle.infolist():
        destination = (source_root / member.filename).resolve()
        if not destination.is_relative_to(source_root.resolve()):
            raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
    bundle.extractall(source_root)
projects = [path.parent for path in source_root.rglob("pyproject.toml")]
if len(projects) != 1:
    raise RuntimeError(f"Expected one project root, found {projects}")
PROJECT = projects[0]
os.chdir(PROJECT)
print(PROJECT)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[test,gpu]"], check=True)

In [ ]:
from routetune.training import environment_report, verify_gpu_environment
print(environment_report())
print(verify_gpu_environment())
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

Use Google Drive for durable run evidence. If the runtime is interrupted before the adapter is saved, the manifest remains `RUNNING` and must not be presented as completed or silently resumed.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
CACHE = Path("/content/routetune-data")
RUNS = Path("/content/drive/MyDrive/RouteTune/phase3-runs")
RUNS.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, "-m", "routetune.data", "--cache-dir", str(CACHE)], check=True)
from routetune.training import prepare_phase3_run
RUN = prepare_phase3_run(CACHE, RUNS, offline=True)
print(RUN)

In [ ]:
from routetune.training import execute_phase3_training
execute_phase3_training(RUN, CACHE)

In [ ]:
from routetune.experiment import load_manifest
import json
manifest = load_manifest(RUN)
print(json.dumps(manifest, indent=2, sort_keys=True))
for name in ("untuned-validation.json", "adapted-validation.json"):
    path = RUN / name
    if path.exists():
        result = json.loads(path.read_text())
        print(name, json.dumps(result["metrics"], indent=2))

## One-time official held-out gate
Leave this disabled until the manifest above is `VERIFIED` and you accept consuming the single official held-out evaluation. The function atomically claims the gate before reading predictions, and refuses a rerun.

In [ ]:
RUN_FINAL_HELDOUT = False
if RUN_FINAL_HELDOUT:
    from routetune.training import execute_final_heldout
    result_path = execute_final_heldout(RUN, CACHE)
    print(result_path.read_text())
else:
    print("Held-out gate not consumed.")

In [ ]:
import shutil
export = shutil.make_archive(f"/content/{RUN.name}", "zip", RUN.parent, RUN.name)
files.download(export)